> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 3 · Notebook 06 — Objects, domain models and design patterns

**Sessions:** S9–S12 (OOP and domain modelling) · S13–S16 (SOLID, patterns, UML) · [Lesson plan](../../docs/lessons/PART_03_PYTHON_ENGINEERING.md) · graded labs in [`labs/part03/`](../../labs/part03/)

**You will:**
1. Give a value object behaviour with dunder methods (`Money`).
2. Expose derived values as properties (`Position`).
3. Enforce an order's life cycle with a state machine.
4. See the Observer and Strategy patterns decouple a trading system.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p3lib.py is in notebooks/part03/
    sys.path.insert(0, str(d))
from decimal import Decimal
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p3lib as p

p.use_course_style()

## 1. A value object: `Money`

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str = "USD"

    def __add__(self, other: "Money") -> "Money":
        if other.currency != self.currency:
            raise ValueError(f"cannot add {self.currency} and {other.currency}")
        return Money(self.amount + other.amount, self.currency)

total = Money(Decimal("1.10")) + Money(Decimal("2.20"))
try:
    Money(Decimal("1"), "USD") + Money(Decimal("1"), "EUR")
    raised = False
except ValueError:
    raised = True
result = p.check("Money addition", (getattr(total, "amount", total), raised), (Decimal("3.30"), True))
result

## 2. Properties: derived values that are never stale

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class Position:
    def __init__(self, symbol):
        self.symbol, self.qty, self.cost = symbol, 0, Decimal("0")

    def buy(self, qty, price):
        self.qty += qty
        self.cost += qty * price

    @property
    def avg_price(self) -> Decimal:
        return self.cost / self.qty if self.qty else Decimal("0")

    def unrealized(self, mark: Decimal) -> Decimal:
        return self.qty * (mark - self.avg_price) if self.qty else Decimal("0")

    def __repr__(self):
        return f"Position({self.symbol}, qty={self.qty}, avg={self.avg_price})"

pos = Position("SPY")
pos.buy(100, Decimal("500"))
pos.buy(300, Decimal("510"))
avg = p.check("average price property", pos.avg_price, Decimal("507.5"))
print(pos, "unrealized at 512:", pos.unrealized(Decimal("512")) if isinstance(pos.avg_price, Decimal) else "…")

## 3. The order life cycle as a state machine

Illegal transitions (a fill after a cancel, a second fill after FILLED) are bugs or broker surprises: refuse them loudly.

In [ ]:
pd.DataFrame([(s, ", ".join(sorted(t)) or "— (final)") for s, t in p.ORDER_TRANSITIONS.items()],
             columns=["state", "may go to"]).set_index("state")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def replay(events):
    state = "NEW"
    for e in events:
        if e not in p.ORDER_TRANSITIONS[state]:
            raise ValueError(f"{state} -> {e} is not allowed")
        state = e
    return state

def outcome(events):
    try:
        return replay(events)
    except ValueError:
        return "error"

seqs = [["SUBMITTED", "PARTIAL", "PARTIAL", "FILLED"], ["SUBMITTED", "CANCELLED", "FILLED"],
        ["FILLED"], ["SUBMITTED", "REJECTED"], ["SUBMITTED", "FILLED", "PARTIAL"]]
states = p.check("order state machine", [outcome(s) for s in seqs], ["FILLED", "error", "error", "REJECTED", "error"])
states

## 4. Observer and Strategy: plug parts in without editing the core

In [ ]:
from collections import defaultdict
from typing import Callable, Protocol

class EventBus:                                   # Observer: publishers don't know their subscribers
    def __init__(self):
        self.subs: dict[str, list[Callable]] = defaultdict(list)
    def subscribe(self, topic, fn):
        self.subs[topic].append(fn)
    def publish(self, topic, payload):
        for fn in self.subs[topic]:
            fn(payload)

class FillModel(Protocol):                        # Strategy: any object with this method will do
    def fill_price(self, side: str, next_open: Decimal) -> Decimal: ...

class NextOpen:
    def fill_price(self, side, next_open):
        return next_open

class NextOpenWithSlippage:
    def __init__(self, bps: Decimal):
        self.bps = bps
    def fill_price(self, side, next_open):
        sign = 1 if side == "BUY" else -1
        return (next_open * (1 + sign * self.bps / 10_000)).quantize(Decimal("0.01"))

journal, alerts = [], []
bus = EventBus()
bus.subscribe("fill", journal.append)
bus.subscribe("fill", lambda f: alerts.append(f) if f["qty"] >= 1000 else None)
for model in (NextOpen(), NextOpenWithSlippage(Decimal("5"))):
    for qty in (100, 2000):
        bus.publish("fill", {"model": type(model).__name__, "qty": qty,
                             "price": model.fill_price("BUY", Decimal("512.00"))})
print(f"journal: {len(journal)} fills, alerts: {len(alerts)} large fills")
pd.DataFrame(journal)

## Questions
1. `Money` is frozen. What bugs does immutability prevent in a multi-strategy system?
2. Why is `avg_price` a property and not an attribute updated by hand?
3. Which SOLID principle lets you add a new fill model without touching the backtester?

**Graded version:** `labs/part03/week09_domain` (Money, Position, property tests) and `labs/part03/week10_patterns`.